# 22 · Feature store por hexágono (G-I3; C2, C8, C15)

O que se sabe de cada hexágono antes do primeiro reporte.

**Faces e setores no plano de coleta:** a face de quadra não tem geometria. Os atributos são levados à rua (código + bairro) e repartidos pelos hexágonos pelo comprimento da rua; o setor de coleta é lido no centro do hexágono. São o plano oficial, de data desconhecida.

**Alagamento:** não há camada de alagamento da cidade. A Defesa Civil tem 0,13% de registros de alagamento; eles não viram feature espacial.

**Regime territorial (morro × planície):** regra provisória por evidências do próprio hexágono (coleta manual, escadaria, rua R3/R4, CIS em Colinas × faixa de rio, CIS em Planície ou Baixo Estuário). **Sem evidência, o hexágono fica `indefinido`** e usa pesos neutros.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "src" / "wastestack").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
from wastestack.arquivos import ler
from wastestack.config import SILVER, GOLD

In [2]:
from wastestack.gold import features_h3
resumo = features_h3()
resumo

{
 "hexagonos": 1943,
 "com_via": 1514,
 "regime": {
  "indefinido": 763,
  "morro": 634,
  "planicie": 546
 },
 "regime_entre_hex_com_via": {
  "morro": 631,
  "indefinido": 488,
  "planicie": 395
 },
 "ruas_r3r4_casadas": 1435,
 "ruas_r3r4_total": 3587,
 "hex_urbanos_sem_drenagem_por_rpa_pct": {
  "1.0": 9.0,
  "2.0": 6.2,
  "3.0": 56.0,
  "4.0": 16.9,
  "5.0": 16.4,
  "6.0": 8.3
 },
 "hex_sem_seletiva_pct": 47.7,
 "hex_com_polo_300m": 115
}


In [3]:
f = ler("motor_features_h3", camada=GOLD)
assert f.h3.is_unique and set(f.regime) <= {"morro", "planicie", "indefinido"}
print("Regime entre hexágonos com via:", resumo["regime_entre_hex_com_via"])
print("Ruas R3/R4 da Defesa Civil casadas com a malha: %d de %d" % (resumo["ruas_r3r4_casadas"], resumo["ruas_r3r4_total"]))
pd.crosstab(f[f.via_m > 0].regime, f[f.via_m > 0].regime_confianca)

Regime entre hexágonos com via: {'morro': 631, 'indefinido': 488, 'planicie': 395}
Ruas R3/R4 da Defesa Civil casadas com a malha: 1435 de 3587
regime_confianca  maior  menor  sem base
regime                                  
indefinido            0      0       488
morro               289    342         0
planicie             99    296         0


## Limites e cuidados

- Não há dado de relevo: o regime é inferido e marcado com confiança. 40% das ruas R3/R4 casam com a malha; o resto não entra.
- Distâncias a pontos de descarte usam listas históricas e linha reta.
- Redes: o comprimento vai para o hexágono do ponto médio do trecho.